In [ ]:
import os
import csv
import subprocess
from pathlib import Path

def get_video_info(video_path: Path):
    """使用 ffprobe 获取视频分辨率、帧率、时长、比特率"""
    cmd = [
        'ffprobe',
        '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries',
        'stream=width,height,r_frame_rate,bit_rate',
        '-show_entries',
        'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(video_path)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        lines = result.stdout.strip().split('\n')
        width = lines[0]
        height = lines[1]
        framerate = eval(lines[2]) if '/' in lines[2] else float(lines[2])
        bitrate = int(lines[3]) if lines[3].isdigit() else 0
        duration = float(lines[4])
        resolution = f"{width}x{height}"
        return resolution, duration, round(framerate, 2), round(bitrate / 1000), 0  # 使用次数初始为0
    except Exception as e:
        raise RuntimeError(f"[错误] 获取视频信息失败：{video_path.name} -> {e}")

def safe_str(s):
    """防止写入CSV时报错，替换无法编码的字符"""
    return s.encode('utf-8', errors='replace').decode('utf-8')

def scan_video_folders(root_dir: Path, output_csv: Path, error_csv: Path):
    """扫描所有视频文件，写入正常和异常表"""
    with open(output_csv, 'w', newline='', encoding='utf-8') as main_file, \
         open(error_csv, 'w', newline='', encoding='utf-8', errors='replace') as error_file:

        main_writer = csv.writer(main_file)
        error_writer = csv.writer(error_file)

        headers = ["子文件夹", "文件名", "分辨率", "时长（秒）", "帧率", "比特率（kbps）", "使用次数"]
        main_writer.writerow(headers)
        error_writer.writerow(headers)

        for subdir, _, files in os.walk(root_dir):
            subfolder = Path(subdir).relative_to(root_dir)
            for file in files:
                if file.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.flv')):
                    video_path = Path(subdir) / file
                    try:
                        resolution, duration, fps, bitrate, usage = get_video_info(video_path)
                        main_writer.writerow([str(subfolder), file, resolution, duration, fps, bitrate, usage])
                        print(f"[✓] {subfolder}/{file} -> {resolution}, {duration:.1f}s, {fps}fps, {bitrate}kbps")
                    except Exception as e:
                        # 写入异常表，字符不合法的替换为 �
                        error_writer.writerow([
                            safe_str(str(subfolder)),
                            safe_str(file),
                            "unknown", 0, 0.0, 0, 0
                        ])
                        print(f"[×] {subfolder}/{file} -> 异常，写入错误表")

    print(f"\n✅ 正常视频信息已保存至：{output_csv}")
    print(f"⚠️ 异常视频信息已保存至：{error_csv}")

# 示例调用
if __name__ == "__main__":
    input_dir = Path("")  # <-- 替换成你的文件夹
    output_csv = input_dir / "video_info.csv"
    error_csv = input_dir / "video_errors.csv"
    scan_video_folders(input_dir, output_csv, error_csv)
